# How to get software on the SCRTP system

Documentation page - [Accessing and installing software](https://docs.scrtp.warwick.ac.uk/general-pages/software.html)

## Understanding Environments

Computing environments is the name we give to a collection of known software that we will use together to run some code or software program.
An environment will not use all the software on the machine, in fact there is a good chance you will have multiple versions of the same software on one machine.

To build an environment you need to isolate which programs and software you need.
To do this we will manipulate the available paths.
We discussed your PATH; it is a list of directories for the computer to search for programs.
There are other paths, for example: PYTHONPATH being a good example for python environment creation.

>> ### Thought Experiment
>>
>> If you are not using an environment then you likely do not know what programs could be being used.
>> However, thinking about versions really highlights how things can go wrong.
>>
>> Consider two packages A and B where B has A as a dependency.
>>
>> #### Without environments
>>
>> You have two projects 1 and 2.
>> 
>> You use A(V1) to generate a result for project 1.
>> 
>> For project 2 you install B to your computer.
>> B however requires a newer version of A(V2) so updates it.
>> 
>> During a review you need to recreate a result from project 1, however, because A has been updated to V2 by B the result is different because of some change made between A(V1) and A(V2).
>> 
>> #### With environments
>>
>> You have two projects 1 and 2.
>> 
>> You create a blank environment for project 1 and install A(V1) in it.
>> A result is created in this environment for project 1.
>>
>> You start a new project 2, creating a new blank environment for it.
>> In this environment you install B. It looks for A but as the environment is blank it can't find it and installs A(V2) which is its dependency.
>> You create a result for project 2.
>> 
>> During a review you need to recreate a result from project 1. 
>> Using the environment you created for project 1 you can recreate the result using A(V1) which has not been changed.
>> 
>> #### Conclusions
>>
>> This is example is slightly reductive and the results are usually more subtle.
>> But it should highlight how things can go wrong. Most Python environments have tens of packages which can all have complicated inter-dependencies on each other.
>> Package/environment managers have been built to manage this so we should use them.




## Modules

On the SCRTP systems we use the modules package.
Modules allows you to create an environment that contains all the software you need without the software you don't.
We have installed (or will install) the software you need and in many cases we install many many versions of common software. 
Lets see what we have:

```bash
module avail
```

or to search for something

```bash
module spider python
```

Modules is going to edit your environment path(s). By editing your paths we can point it to the correct version using your path.

Let's try and load Python for you to use, as opposed to system Python in `/usr/bin/`.
> Don't use the Python in /usr/bin/python. This is the system Python that the system uses and it does not have the ability to be extended with packages or used with other modules in the way you will want to use it!
> The system Python may also be updated or modified during system maintenance
> You can tell which Python is loaded using `which python`.

```bash 
module load Python/3.11.5
```
However, this throws an error as Python has dependencies.
Furthermore different versions of Python will have different dependencies.
We can use `module spider` again to query this.

```bash
module spider Python/3.11.5
```

```bash
module spider Python/3.9.6
```

For 3.11.5 we need GCCcore/13.2.0 loaded and for 3.9.6 we need GCCcore/11.2.0

So to load Python/3.11.5 we run

```bash
module load GCCcore/13.2.0 Python/3.11.5
```

And now we have Python.

```bash
which python
```

And we can see all the modules we have loaded.

```bash
module list
```

Now while you can put modules to load in your .bash_profile this isn't recommended.
You should instead simplify loading and unloading using the module collections feature.
To save this Python setup for use later we can create a collection using `module save`.

```bash
module save py3_11_env
```

Then next time you log onto the system you can quickly run.

```bash
module restore py3_11_env
```

## Now let's talk about building on-top of modules

The modules system is the way you should bring in programs that need to have been specifically compiled to run on the hardware, e.g., BLAS. However some language packages will use these as system dependencies.

### An example in Python

> Upfront, if you can avoid it don't use Conda/Miniconda as it plays much less nicely with building clean environments, if you _have_ to use it then do not use any modules-provided Python.

The steps you should follow:

1. Load all your module dependencies using `module` load ensure compatibility.
2. Create a collection using `module save module_<env_name>`, you can name it however you like.
3. Create a Python environment using `python -m venv <env_name>`, again name it how you like.
4. Activate the env, `source <env_name>/bin/activate`
5. Install additional dependencies using `pip`, `pip install ____`
6. Check all expected libraries are available using `pip list`


> How to Debug: Deactivate env and unload modules, `deactivate` and `module purge`, then restore modules and source env `module restore <name>` and  reactivate env `source <name>/bin/activate`.
>
> Note: Do not load modules in the Python environment, if a new system module is desired the Python env must be recreated from scratch.
>

Extra considerations, some Python packages such as PyTorch have been specifically compiled to be optimized for our hardware.
Additionally, some bundles are provided for a quick `one and done` software stack in Python.
If you want to use these then you load them before creating your Python environment.
Finally if a Python package, when installed by pip, has a system dependency that can be provided by modules this is advisable.

## Exercise: load a Python environment

This is a practice session for using these tools. 
If you are attending this course in person then you should also attempt to build an environment you will use in your research so we can help you.
If you are not in person consider using the SCRTP slack or, during semester, our [Drop In session](https://warwick.ac.uk/research/rtp/sc/user_support/research-computing-drop-in/) to come and get advice on building your environment.

We want an environment with the following spec.

1. Python (modules latest possible)
2. PyTorch (from modules PyTorch)
3. Jupyter Notebook (from modules)
4. Scikit-learn (from pip)
5. matplotlib (from pip)


To validate this we will later queue a test using SLURM.

First, we make sure that no modules are loaded that could interfer by purging all loaded modules:
```bash
[me@godzilla.csc.warwick.ac.uk ~]$ module purge
```
Then, we can load the modules from the modules system:

```bash
[me@godzilla.csc.warwick.ac.uk ~]$ module load GCC/11.3.0  OpenMPI/4.1.4 PyTorch/1.12.1 JupyterLab/3.5.0

```

Note that by loading PyTorch, a compatible Python version has been loaded automatically:

```bash
[me@godzilla.csc.warwick.ac.uk ~]$ which python
/software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/bin/python
```

If this does not happen automatically, you will need to check using `module spider` which available Python version has the same dependencies (e.g., GCC/11.3.0) that you already loaded.

```bash
[me@godzilla.csc.warwick.ac.uk ~]$ module list

Currently Loaded Modules:
  1) GCCcore/11.3.0     19) FFTW/3.3.10             37) MPFR/4.1.0           55) lz4/1.9.3
  2) zlib/1.2.12        20) FFTW.MPI/3.3.10         38) NASM/2.15.05         56) zstd/1.5.2
  3) binutils/2.38      21) ScaLAPACK/2.2.0-fb      39) x264/20220620        57) libdeflate/1.10
  4) GCC/11.3.0         22) Ninja/1.10.2            40) LAME/3.100           58) LibTIFF/4.3.0
  5) numactl/2.0.14     23) bzip2/1.0.8             41) x265/3.5             59) Pillow/9.1.1
  6) XZ/5.2.5           24) ncurses/6.3             42) expat/2.4.8          60) expecttest/0.1.3
  7) libxml2/2.9.13     25) libreadline/8.1.2       43) libpng/1.6.37        61) PyTorch/1.12.1
  8) libpciaccess/0.16  26) Tcl/8.6.12              44) Brotli/1.0.9         62) OpenPGM/5.2.122
  9) hwloc/2.7.1        27) SQLite/3.38.3           45) freetype/2.12.1      63) libsodium/1.0.18
 10) OpenSSL/1.1        28) GMP/6.2.1               46) util-linux/2.38      64) ZeroMQ/4.3.4
 11) libevent/2.1.12    29) libffi/3.4.2            47) fontconfig/2.14.0    65) libxslt/1.1.34
 12) UCX/1.12.1         30) Python/3.10.4           48) xorg-macros/1.19.3   66) lxml/4.9.1
 13) libfabric/1.15.1   31) protobuf/3.19.4         49) X11/20220504         67) BeautifulSoup/4.10.0
 14) PMIx/4.1.2         32) protobuf-python/3.19.4  50) FriBidi/1.0.12       68) IPython/8.5.0
 15) UCC/1.0.0          33) pybind11/2.9.2          51) FFmpeg/4.4.2         69) jupyter-server/1.21.0
 16) OpenMPI/4.1.4      34) SciPy-bundle/2022.05    52) libjpeg-turbo/2.1.3  70) JupyterLab/3.5.0
 17) OpenBLAS/0.3.20    35) libyaml/0.2.5           53) jbigkit/2.1
 18) FlexiBLAS/3.2.0    36) PyYAML/6.0              54) gzip/1.12
```

```bash
[me@godzilla.csc.warwick.ac.uk ~]$ module save mod_pyt_jpy
Saved current collection of modules to: "mod_pyt_jpy"

[me@godzilla.csc.warwick.ac.uk ~]$ which python
/software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/bin/python
[me@godzilla.csc.warwick.ac.uk ~]$ python -m venv env_pyt_jpy --system-site-packages
[me@godzilla.csc.warwick.ac.uk ~]$ source env_pyt_jpy/bin/activate
```

The Python Path is now tied to the virtual environment:
```bash
(env_pyt_jpy) [me@godzilla.csc.warwick.ac.uk ~]$ which python
/storage/chem/me/env_pyt_ase_jpy/bin/python
```

We can look at which Python packages are already available.
```bash
(env_pyt_jpy) [me@godzilla.csc.warwick.ac.uk ~]$ pip list
Package                           Version
--------------------------------- -----------
alabaster                         0.7.12
anyio                             3.6.1
appdirs                           1.4.4
argon2-cffi                       20.1.0
ase                               3.25.0b1
asn1crypto                        1.5.1
asttokens                         2.0.8
async-generator                   1.10
atomicwrites                      1.4.0
attrs                             21.4.0
Babel                             2.10.1
backcall                          0.2.0
backports.entry-points-selectable 1.1.1
backports.functools-lru-cache     1.6.4
bcrypt                            3.2.2
beautifulsoup4                    4.10.0
beniget                           0.4.1
bitstring                         3.1.9
bleach                            5.0.1
blist                             1.3.6
Bottleneck                        1.3.4
CacheControl                      0.12.11
cachy                             0.3.0
certifi                           2021.10.8
cffi                              1.15.0
chardet                           4.0.0
charset-normalizer                2.0.12
cleo                              0.8.1
click                             8.1.3
clikit                            0.6.2
colorama                          0.4.4
contourpy                         1.1.1
crashtest                         0.3.1
cryptography                      37.0.1
cycler                            0.12.1
Cython                            0.29.28
deap                              1.3.3
debugpy                           1.4.1
decorator                         5.1.1
defusedxml                        0.7.1
deprecation                       2.1.0
distlib                           0.3.4
docopt                            0.6.2
docutils                          0.17.1
ecdsa                             0.17.0
editables                         0.3
entrypoints                       0.4
executing                         1.0.0
expecttest                        0.1.3
fastjsonschema                    2.16.1
filelock                          3.6.0
flit                              3.7.1
flit_core                         3.7.1
fonttools                         4.43.1
fsspec                            2022.3.0
future                            0.18.2
gast                              0.5.3
glob2                             0.7
html5lib                          1.1
idna                              3.3
imagesize                         1.3.0
importlib-metadata                4.11.3
importlib-resources               5.7.1
iniconfig                         1.1.1
intervaltree                      3.1.0
intreehooks                       1.0
ipaddress                         1.0.23
ipykernel                         6.13.0
ipython                           8.5.0
ipython-genutils                  0.2.0
ipywidgets                        7.6.3
jedi                              0.18.1
jeepney                           0.8.0
Jinja2                            3.1.2
joblib                            1.1.0
json5                             0.9.10
jsonschema                        4.4.0
jupyter-client                    7.3.1
jupyter-core                      4.10.0
jupyter-packaging                 0.12.0
jupyter-server                    1.21.0
jupyterlab                        3.5.0
jupyterlab-pygments               0.1.2
jupyterlab-server                 2.13.0
jupyterlab-widgets                3.0.3
keyring                           23.5.0
keyrings.alt                      4.1.0
kiwisolver                        1.4.5
liac-arff                         2.5.0
lockfile                          0.12.2
lxml                              4.9.1
MarkupSafe                        2.1.1
matplotlib                        3.8.0
matplotlib-inline                 0.1.2
mistune                           0.8.1
mock                              4.0.3
more-itertools                    8.12.0
mpi4py                            3.1.3
mpmath                            1.2.1
msgpack                           1.0.3
nbclassic                         0.4.8
nbclient                          0.6.3
nbconvert                         6.5.3
nbformat                          5.4.0
nest-asyncio                      1.5.5
netaddr                           0.8.0
netifaces                         0.11.0
notebook                          6.4.0
notebook-shim                     0.1.0
numexpr                           2.8.1
numpy                             1.22.3
packaging                         20.9
pandas                            1.4.2
pandocfilters                     1.5.0
paramiko                          2.10.4
parso                             0.8.3
pastel                            0.2.1
pathlib2                          2.3.7.post1
pathspec                          0.9.0
pbr                               5.8.1
pexpect                           4.8.0
pickleshare                       0.7.5
Pillow                            9.1.1
pip                               22.0.4
pkginfo                           1.8.2
platformdirs                      2.4.1
pluggy                            1.0.0
ply                               3.11
poetry                            1.1.13
poetry-core                       1.0.8
prometheus-client                 0.11.0
prompt-toolkit                    3.0.31
protobuf                          3.19.4
psutil                            5.9.0
ptyprocess                        0.7.0
pure-eval                         0.2.2
py                                1.11.0
py-expression-eval                0.3.14
pyasn1                            0.4.8
pybind11                          2.9.2
pycparser                         2.21
pycryptodome                      3.17
Pygments                          2.12.0
pylev                             1.4.0
PyNaCl                            1.5.0
pyparsing                         3.0.8
pyrsistent                        0.18.1
pytest                            7.1.2
python-dateutil                   2.8.2
pythran                           0.11.0
pytoml                            0.1.21
pytz                              2022.1
PyYAML                            6.0
pyzmq                             23.2.1
regex                             2022.4.24
requests                          2.27.1
requests-toolbelt                 0.9.1
scandir                           1.10.0
SciPy                             1.8.1
SecretStorage                     3.3.2
semantic-version                  2.9.0
Send2Trash                        1.8.0
setuptools                        58.1.0
setuptools-rust                   1.3.0
setuptools-scm                    6.4.2
shellingham                       1.4.0
simplegeneric                     0.8.1
simplejson                        3.17.6
six                               1.16.0
sniffio                           1.3.0
snowballstemmer                   2.2.0
sortedcontainers                  2.4.0
soupsieve                         2.3.1
Sphinx                            4.5.0
sphinx-bootstrap-theme            0.8.1
sphinxcontrib-applehelp           1.0.2
sphinxcontrib-devhelp             1.0.2
sphinxcontrib-htmlhelp            2.0.0
sphinxcontrib-jsmath              1.0.1
sphinxcontrib-qthelp              1.0.3
sphinxcontrib-serializinghtml     1.1.5
sphinxcontrib-websupport          1.2.4
stack-data                        0.5.0
tabulate                          0.8.9
terminado                         0.13.0
testpath                          0.6.0
threadpoolctl                     3.1.0
tinycss2                          1.1.1
toml                              0.10.2
tomli                             2.0.1
tomli_w                           1.0.0
tomlkit                           0.10.2
torch                             1.12.1
tornado                           6.2
traitlets                         5.2.0
typing_extensions                 4.2.0
ujson                             5.2.0
urllib3                           1.26.9
virtualenv                        20.14.1
wcwidth                           0.2.5
webencodings                      0.5.1
websocket-client                  1.4.2
wheel                             0.37.1
widgetsnbextension                3.5.1
xlrd                              2.0.1
zipfile36                         0.1.3
zipp                              3.8.0
```
You can now install the rest of the packages via pip

```bash
(env_pyt_jpy) [me@godzilla.csc.warwick.ac.uk ~]$ pip install scikit-learn
Collecting scikit-learn
  Downloading scikit_learn-1.7.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 78.9 MB/s eta 0:00:00
Requirement already satisfied: scipy>=1.8.0 in /software/easybuild/software/SciPy-bundle/2022.05-foss-2022.05/lib/python3.10/site-packages (from scikit-learn) (1.8.1)
Requirement already satisfied: threadpoolctl>=3.1.0 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from scikit-learn) (3.1.0)
Requirement already satisfied: numpy>=1.22.0 in /software/easybuild/software/SciPy-bundle/2022.05-foss-2022.05/lib/python3.10/site-packages (from scikit-learn) (1.22.3)
Collecting joblib>=1.2.0
  Downloading joblib-1.5.1-py3-none-any.whl (307 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.7/307.7 KB 40.4 MB/s eta 0:00:00
Installing collected packages: joblib, scikit-learn
  Attempting uninstall: joblib
    Found existing installation: joblib 1.1.0
    Not uninstalling joblib at /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages, outside environment /storage/chem/me/env_pyt_ase_jpy
    Can't uninstall 'joblib'. No files were found to uninstall.
Successfully installed joblib-1.5.1 scikit-learn-1.7.1
```

```bash
(env_pyt_jpy) [me@godzilla.csc.warwick.ac.uk ~]$ pip install seaborn
Collecting seaborn
  Downloading seaborn-0.13.2-py3-none-any.whl (294 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 13.0 MB/s eta 0:00:00
Requirement already satisfied: pandas>=1.2 in /software/easybuild/software/SciPy-bundle/2022.05-foss-2022.05/lib/python3.10/site-packages (from seaborn) (1.4.2)
Requirement already satisfied: numpy!=1.24.0,>=1.20 in /software/easybuild/software/SciPy-bundle/2022.05-foss-2022.05/lib/python3.10/site-packages (from seaborn) (1.22.3)
Requirement already satisfied: matplotlib!=3.6.1,>=3.4 in /home/chem/iasfjh/.local/lib/python3.10/site-packages (from seaborn) (3.8.0)
Requirement already satisfied: pyparsing>=2.3.1 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (3.0.8)
Requirement already satisfied: cycler>=0.10 in /home/chem/iasfjh/.local/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (0.12.1)
Requirement already satisfied: fonttools>=4.22.0 in /home/chem/iasfjh/.local/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (4.43.1)
Requirement already satisfied: pillow>=6.2.0 in /software/easybuild/software/Pillow/9.1.1-GCCcore-11.3.0/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (9.1.1)
Requirement already satisfied: python-dateutil>=2.7 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (2.8.2)
Requirement already satisfied: kiwisolver>=1.0.1 in /home/chem/iasfjh/.local/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (1.4.5)
Requirement already satisfied: packaging>=20.0 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (20.9)
Requirement already satisfied: contourpy>=1.0.1 in /home/chem/iasfjh/.local/lib/python3.10/site-packages (from matplotlib!=3.6.1,>=3.4->seaborn) (1.1.1)
Requirement already satisfied: pytz>=2020.1 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from pandas>=1.2->seaborn) (2022.1)
Requirement already satisfied: six>=1.5 in /software/easybuild/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/site-packages (from python-dateutil>=2.7->matplotlib!=3.6.1,>=3.4->seaborn) (1.16.0)
Installing collected packages: seaborn
Successfully installed seaborn-0.13.2
```

```bash
(env_pyt_jpy) [me@godzilla.csc.warwick.ac.uk ~]$ pip list
Package                           Version
--------------------------------- -----------
alabaster                         0.7.12
anyio                             3.6.1
appdirs                           1.4.4
argon2-cffi                       20.1.0
ase                               3.25.0b1
asn1crypto                        1.5.1
asttokens                         2.0.8
async-generator                   1.10
atomicwrites                      1.4.0
attrs                             21.4.0
Babel                             2.10.1
backcall                          0.2.0
backports.entry-points-selectable 1.1.1
backports.functools-lru-cache     1.6.4
bcrypt                            3.2.2
beautifulsoup4                    4.10.0
beniget                           0.4.1
bitstring                         3.1.9
bleach                            5.0.1
blist                             1.3.6
Bottleneck                        1.3.4
CacheControl                      0.12.11
cachy                             0.3.0
certifi                           2021.10.8
cffi                              1.15.0
chardet                           4.0.0
charset-normalizer                2.0.12
cleo                              0.8.1
click                             8.1.3
clikit                            0.6.2
colorama                          0.4.4
contourpy                         1.1.1
crashtest                         0.3.1
cryptography                      37.0.1
cycler                            0.12.1
Cython                            0.29.28
deap                              1.3.3
debugpy                           1.4.1
decorator                         5.1.1
defusedxml                        0.7.1
deprecation                       2.1.0
distlib                           0.3.4
docopt                            0.6.2
docutils                          0.17.1
ecdsa                             0.17.0
editables                         0.3
entrypoints                       0.4
executing                         1.0.0
expecttest                        0.1.3
fastjsonschema                    2.16.1
filelock                          3.6.0
flit                              3.7.1
flit_core                         3.7.1
fonttools                         4.43.1
fsspec                            2022.3.0
future                            0.18.2
gast                              0.5.3
glob2                             0.7
html5lib                          1.1
idna                              3.3
imagesize                         1.3.0
importlib-metadata                4.11.3
importlib-resources               5.7.1
iniconfig                         1.1.1
intervaltree                      3.1.0
intreehooks                       1.0
ipaddress                         1.0.23
ipykernel                         6.13.0
ipython                           8.5.0
ipython-genutils                  0.2.0
ipywidgets                        7.6.3
jedi                              0.18.1
jeepney                           0.8.0
Jinja2                            3.1.2
joblib                            1.5.1
json5                             0.9.10
jsonschema                        4.4.0
jupyter-client                    7.3.1
jupyter-core                      4.10.0
jupyter-packaging                 0.12.0
jupyter-server                    1.21.0
jupyterlab                        3.5.0
jupyterlab-pygments               0.1.2
jupyterlab-server                 2.13.0
jupyterlab-widgets                3.0.3
keyring                           23.5.0
keyrings.alt                      4.1.0
kiwisolver                        1.4.5
liac-arff                         2.5.0
lockfile                          0.12.2
lxml                              4.9.1
MarkupSafe                        2.1.1
matplotlib                        3.8.0
matplotlib-inline                 0.1.2
mistune                           0.8.1
mock                              4.0.3
more-itertools                    8.12.0
mpi4py                            3.1.3
mpmath                            1.2.1
msgpack                           1.0.3
nbclassic                         0.4.8
nbclient                          0.6.3
nbconvert                         6.5.3
nbformat                          5.4.0
nest-asyncio                      1.5.5
netaddr                           0.8.0
netifaces                         0.11.0
notebook                          6.4.0
notebook-shim                     0.1.0
numexpr                           2.8.1
numpy                             1.22.3
packaging                         20.9
pandas                            1.4.2
pandocfilters                     1.5.0
paramiko                          2.10.4
parso                             0.8.3
pastel                            0.2.1
pathlib2                          2.3.7.post1
pathspec                          0.9.0
pbr                               5.8.1
pexpect                           4.8.0
pickleshare                       0.7.5
Pillow                            9.1.1
pip                               22.0.4
pkginfo                           1.8.2
platformdirs                      2.4.1
pluggy                            1.0.0
ply                               3.11
poetry                            1.1.13
poetry-core                       1.0.8
prometheus-client                 0.11.0
prompt-toolkit                    3.0.31
protobuf                          3.19.4
psutil                            5.9.0
ptyprocess                        0.7.0
pure-eval                         0.2.2
py                                1.11.0
py-expression-eval                0.3.14
pyasn1                            0.4.8
pybind11                          2.9.2
pycparser                         2.21
pycryptodome                      3.17
Pygments                          2.12.0
pylev                             1.4.0
PyNaCl                            1.5.0
pyparsing                         3.0.8
pyrsistent                        0.18.1
pytest                            7.1.2
python-dateutil                   2.8.2
pythran                           0.11.0
pytoml                            0.1.21
pytz                              2022.1
PyYAML                            6.0
pyzmq                             23.2.1
regex                             2022.4.24
requests                          2.27.1
requests-toolbelt                 0.9.1
scandir                           1.10.0
scikit-learn                      1.7.1
SciPy                             1.8.1
seaborn                           0.13.2
SecretStorage                     3.3.2
semantic-version                  2.9.0
Send2Trash                        1.8.0
setuptools                        58.1.0
setuptools-rust                   1.3.0
setuptools-scm                    6.4.2
shellingham                       1.4.0
simplegeneric                     0.8.1
simplejson                        3.17.6
six                               1.16.0
sniffio                           1.3.0
snowballstemmer                   2.2.0
sortedcontainers                  2.4.0
soupsieve                         2.3.1
Sphinx                            4.5.0
sphinx-bootstrap-theme            0.8.1
sphinxcontrib-applehelp           1.0.2
sphinxcontrib-devhelp             1.0.2
sphinxcontrib-htmlhelp            2.0.0
sphinxcontrib-jsmath              1.0.1
sphinxcontrib-qthelp              1.0.3
sphinxcontrib-serializinghtml     1.1.5
sphinxcontrib-websupport          1.2.4
stack-data                        0.5.0
tabulate                          0.8.9
terminado                         0.13.0
testpath                          0.6.0
threadpoolctl                     3.1.0
tinycss2                          1.1.1
toml                              0.10.2
tomli                             2.0.1
tomli_w                           1.0.0
tomlkit                           0.10.2
torch                             1.12.1
tornado                           6.2
traitlets                         5.2.0
typing_extensions                 4.2.0
ujson                             5.2.0
urllib3                           1.26.9
virtualenv                        20.14.1
wcwidth                           0.2.5
webencodings                      0.5.1
websocket-client                  1.4.2
wheel                             0.37.1
widgetsnbextension                3.5.1
xlrd                              2.0.1
zipfile36                         0.1.3
zipp                              3.8.0
```


To test the script you will need to run the following:

test_script.py
```python
try:
    import sklearn
    print("scikit-learn available")
except ImportError:
    print("scikit-learn not available")
    raise
try:
    import torch
    print("torch reports {torch.cuda.is_available()}")
except ImportError:
    print("torch not available")
    raise
try:
    import seaborn
    print("seaborn available")
except ImportError:
    print("seaborn not available")
    raise
```

Either using vim to write it directly or scp to make it locally and copy it over make sure that file is in your home on avon.

Then create this file in your home as well.

The rest of this information is about submitting jobs. More on this later, for now copy and paste verbatim.

test_env.sbatch
```bash
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=3700
#SBATCH --time=00:10:00

module purge && module restore mod_pyt_jpy

source env_pyt_jpy/bin/activate

python test_script.py

jupyter-lab --no-browser
```

Submit this file using:

```bash
sbatch test_env.sbatch
```

You will need to keep an eye on your job like this

```bash
squeue --user=<scrtpUserName> --start
```

Then when the job starts make note of the node and run from a new terminal:

```bash
ssh -J <scrtpUserName>@godzilla.csc.warwick.ac.uk -N -L 8888:127.0.0.1:8888 <scrtpUserName>@taskfarm<node-number>.csc.warwick.ac.uk
```

Then check the output file which will be of the form slurm-xxxxxx.out (where xxxxxx is the SLURM job ID returned when submitting the batch script).
In this file you will find your token (i.e., and internet address that looks something like this: http://localhost:8888/lab?token=<alpha-numeric-string>). Paste the full token into a browser and we can use jupyter.
Launch a notebook, and paste and run the test python code as we used above.

```python
try:
    import sklearn
    print("scikit-learn available")
except ImportError:
    print("scikit-learn not available")
    raise
try:
    import torch
    print("torch reports {torch.cuda.is_available()}")
except ImportError:
    print("torch not available")
    raise
try:
    import seaborn
    print("seaborn available")
except ImportError:
    print("seaborn not available")
    raise
```

If that goes well use ctrl-c to end the notebook and quit to end the HPC job.